In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path
from collections import Counter

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# pytorch modules and dependencies
import torch
import torchvision
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataloader, Dataset

# import resnet50 pretrained model
from torchvision.models import resnet50, ResNet50_Weights

# reads DICOM medical files
import pydicom

# displays graphs and images
import matplotlib.pyplot as plt

### check device if CUDA is available if not use CPU

In [ ]:
device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')
print(device)
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))

# load, explore and analyze the data provided

### load and analyze the train.csv

In [ ]:
ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

# load the train csv data and describe
train = pd.read_csv(ROOT / "train.csv")
print(train.shape)
display(train.head(5))

# get the labels
LABELS = [c for c in train.columns if c not in ("StudyInstanceUID", "Report")]
print("\n", len(LABELS), LABELS)

In [ ]:
# Distinguish fully labelled, partially labelled, and unlabelled studies.
label_values = train[LABELS]
fully_labelled = label_values.notna().all(axis=1)
any_label = label_values.notna().any(axis=1)
partially_labelled = any_label & ~fully_labelled
unlabelled = ~any_label

print("Fully labelled studies:", int(fully_labelled.sum()))
print("Partially labelled studies:", int(partially_labelled.sum()))
print("Studies with no structured labels:", int(unlabelled.sum()))
print("Missing values by label:")
display(
    label_values.isna().sum().sort_values(ascending=False).to_frame("missing_count")
)

In [ ]:
import matplotlib.pyplot as plt

# Label prevalence and imbalance: compute only where the relevant label is known.
known_counts = label_values.notna().sum()
positive_counts = (label_values == 1).sum()
negative_counts = (label_values == 0).sum()
label_audit = pd.DataFrame({
    "known": known_counts,
    "positive": positive_counts,
    "negative": negative_counts,
    "positive_fraction_among_known": positive_counts / known_counts.replace(0, np.nan),
})
display(label_audit.sort_values("positive_fraction_among_known"))
label_audit["positive_fraction_among_known"].sort_values().plot.barh(
    figsize=(8, 6), title="Positive fraction among known labels"
)
plt.tight_layout()

In [ ]:
# explore reports for potential weak-label extraction
print("Missing reports:", int(train["Report"].isna().sum()))
for i in range(min(5, len(train))):
    print(f"\nReport sample {i} | study={train.loc[i, 'StudyInstanceUID']}")
    print(str(train.loc[i, "Report"])[:1000])

### load and analyze the train_series.csv

In [ ]:
# load and describe the train_series data
train_series = pd.read_csv(ROOT / "train_series.csv")
print(train_series.shape)
display(train_series.head(5))
print("\nStudies in train_series table: ", train_series["StudyInstanceUID"].nunique())
print("\nSeries per study summary:")
display(
    train_series.groupby("StudyInstanceUID").size().describe().to_frame("series_per_study")
)

# anatomical plane
print("\nSeries counts by plane:")
display(
    train_series["Anatomical_Plane"].value_counts(dropna=False).to_frame("series_count")
)

In [ ]:
# compare fluid sensitive and fat suppression columns to check for similarity
print(pd.crosstab(train_series["Fluid_Sensitive"], train_series["Fat_Suppression"]))

# check how many studies contain at least on fluid_sensitive series in each plane
fs = train_series[train_series["Fluid_Sensitive"] == 1]
print("\nStudies with fluid_sensitive series by plane:")
print(fs.groupby("Anatomical_Plane")["StudyInstanceUID"].nunique())

### load, explore and analyze the DICOM slices in the train_series folder

In [ ]:
import pydicom

ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

def series_dir(split_dir, study_uid, series_uid):
    return ROOT/split_dir/study_uid/series_uid

sample = train_series.iloc[0]
sample_dir = series_dir("train_series", sample.StudyInstanceUID, sample.SeriesInstanceUID)
sample_files = list(sample_dir.glob("*.dcm"))
print("Example study:", sample.StudyInstanceUID)
print("Example series:", sample.SeriesInstanceUID)
print("DICOM files found:", len(sample_files))
if sample_files:
    ds = pydicom.dcmread(sample_files[0], stop_before_pixels=True)
    for tag in ["Modality", "SeriesDescription", "Rows", "Columns", "SliceThickness",
                "PixelSpacing", "RepetitionTime", "EchoTime", "ImageType"]:
        print(f"{tag:20s}", getattr(ds, tag, "n/a"))

In [ ]:
# load, display and compile a complete series into a single NumPy array
def load_series(dir_path):
    files = list(Path(dir_path).glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"No DICOM slices found in {dir_path}")
        
    dss = [
        pydicom.dcmread(f)
        for f in Path(dir_path).glob("*.dcm")
    ]

    # sort the slices in anatomical order
    try:
        iop = np.asarray(dss[0].ImageOrientationPatient, dtype=np.float64)
        normal = np.cross(iop[:3], iop[3:])

        dss.sort(
            key=lambda x: float(
                np.dot(
                    normal,
                    np.asarray(x.ImagePositionPatient, dtype=np.float64)
                )
            )
        )
    except (AttributeError, ValueError, TypeError):
        dss.sort(
            key=lambda ds: int(
                getattr(x, "InstanceNumber", 0)
            )
        )

    # Apply DICOM rescale where present; handle MONOCHROME1 so intensities have consistent polarity.
    slices = []
    for ds in dss:
        image = ds.pixel_array.astype(np.float32)
        image = image * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            image = image.max() - image
        slices.append(image)
    return np.stack(slices), dss

vol, sample_datasets = load_series(sample_dir)
print("Volume shape (no. of slices, image height, image width: ", vol.shape)

In [ ]:
import matplotlib.pyplot as plt

# displaying a few select slices for visual inspection
n = min(8, len(vol))
idx = np.linspace(0, len(vol) - 1, n).astype(int)
fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.5))
for ax, i in zip(np.atleast_1d(axes), idx):
    ax.imshow(vol[1], cmap="grey");
    ax.set_title(f"slice {i}");
plt.suptitle(f"{sample.Anatomical_Plane} | fluid={sample.Fluid_Sensitive} fat_sup={sample.Fat_Suppression}")
plt.show()

### Build a complete series manifest

In [ ]:
def count_dicom_slices(split_dir, study_uid, series_uid):
    return len(list(series_dir(split_dir, study_uid, series_uid).glob("*.dcm")))

series_manifest = train_series.copy()
series_manifest["n_slices"] = [
    count_dicom_slices("train_series", row.StudyInstanceUID, row.SeriesInstanceUID)
    for row in series_manifest.itertuples(index=False)
]
series_manifest["series_path_exists"] = [
    series_dir("train_series", row.StudyInstanceUID, row.SeriesInstanceUID).is_dir()
    for row in series_manifest.itertuples(index=False)
]

study_manifest = train[["StudyInstanceUID"]].copy()
study_manifest["fully_labelled"] = fully_labelled.to_numpy()
study_manifest["partially_labelled"] = partially_labelled.to_numpy()
study_manifest["unlabelled"] = unlabelled.to_numpy()
study_manifest = study_manifest.merge(
    series_manifest.groupby("StudyInstanceUID").agg(
        n_series=("SeriesInstanceUID", "nunique"),
        total_slices=("n_slices", "sum"),
        missing_series_dirs=("series_path_exists", lambda x: int((~x).sum())),
    ),
    on="StudyInstanceUID", how="left"
)
print("Study inventory:", study_manifest.shape)
display(study_manifest.head())
print("Series directories missing:", int((~series_manifest["series_path_exists"]).sum()))
print("Series with zero slices:", int((series_manifest["n_slices"] == 0).sum()))
print("Slice-count distribution:")
display(series_manifest["n_slices"].describe().to_frame("slices_per_series"))
print("Series counts by plane and fluid-sensitive flag:")
display(pd.crosstab(series_manifest["Anatomical_Plane"], series_manifest["Fluid_Sensitive"], dropna=False))

# Save manifests for reproducibility
series_manifest.to_csv("/kaggle/working/series_manifest.csv", index=False)
study_manifest.to_csv("/kaggle/working/study_manifest.csv", index=False)

# prepare the data for pytorch

In [ ]:
from torch.utils.data import Dataset

def normalize_volume(vol):
    # Per-volume robust scaling
    lo, hi = np.percentile(vol, [0.5, 99.5])
    vol = np.clip(vol, lo, hi)
    return (vol - lo) / max(float(hi - lo), 1e-6)

class KneeStudyDataset(Dataset):
    def __init__(self, study_ids, series_df, split_dir="train_series",
                 labels_df=None):
        self.ids = list(study_ids)
        self.series_df = series_df
        self.split_dir = split_dir
        self.labels = None if labels_df is None else labels_df.set_index("StudyInstanceUID")
        self.series_by_study = {
            sid: g.copy() for sid, g in series_df.groupby("StudyInstanceUID")
        }

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        sid = self.ids[i]
        rows = self.series_by_study.get(sid)
        if rows is None or rows.empty:
            raise RuntimeError(f"No series metadata for study {sid}")

        series_volumes, series_info = [], []
        for row in rows.itertuples(index=False):
            path = series_dir(self.split_dir, sid, row.SeriesInstanceUID)
            if not path.is_dir() or not any(path.glob("*.dcm")):
                raise FileNotFoundError(f"Missing DICOM series directory/files: {path}")
            vol, _ = load_series(path)
            vol = normalize_volume(vol)
            # Keeping native H x W resolution
            vol = torch.from_numpy(vol.astype(np.float32))  # (slices, H, W)
            series_volumes.append(vol)
            series_info.append({
                "series_uid": row.SeriesInstanceUID,
                "plane": getattr(row, "Anatomical_Plane", None),
                "fluid_sensitive": getattr(row, "Fluid_Sensitive", None),
                "fat_suppression": getattr(row, "Fat_Suppression", None),
                "n_slices": int(vol.shape[0]),
            })

        item = {"study_uid": sid, "series": series_volumes, "series_info": series_info}
        if self.labels is not None:
            y = self.labels.loc[sid, LABELS].to_numpy(dtype=np.float32)
            item["y"] = torch.tensor(y, dtype=torch.float32)
            item["label_known"] = torch.tensor(~np.isnan(y), dtype=torch.bool)
        return item

def collate_one_study(batch):
    # Each study has a different number of series/slices, retaining each as a list.
    if len(batch) != 1:
        raise ValueError("Use batch_size=1 with collate_one_study for variable-length studies.")
    return batch[0]

### testing the complete data pipeline

In [ ]:
from torch.utils.data import DataLoader

# Partially labelled studies are retained: label_known marks which targets are usable.
supervised_ids = train.loc[train[LABELS].notna().any(axis=1), "StudyInstanceUID"].tolist()
unlabelled_ids = train.loc[train[LABELS].isna().all(axis=1), "StudyInstanceUID"].tolist()

# Using labels_df=train for partially labelled studies; missing targets are masked by label_known.
ds_train = KneeStudyDataset(
    supervised_ids, series_manifest, split_dir="train_series",
    labels_df=train
)
loader = DataLoader(
    ds_train, batch_size=1, shuffle=True, num_workers=0,
    collate_fn=collate_one_study
)

# Smoke-testing one complete study (all available series, all slices).
batch = next(iter(loader))
print("Study:", batch["study_uid"])
print("Number of series retained:", len(batch["series"]))
print("Slices per series:", [tuple(v.shape) for v in batch["series"]])
print("Labels shape:", tuple(batch["y"].shape))
print("Known labels:", int(batch["label_known"].sum()), "/", len(LABELS))

# Unlabelled studies remain available for future self-supervised/semi-supervised learning.
print("Unlabelled studies available for additional learning:", len(unlabelled_ids))